# Finale Bereinigung des Rohdatensatzes (Abschnitt 4.6 der Arbeit)

Dieses Notebook wendet **nicht** die Benchmark-Stichproben aus Kapitel 4.1-4.4 an,
sondern die dort **jeweils empirisch beste Methode je Fehlertyp** auf den
**vollständigen, echten Rohdatensatz** (`data/rfd_main.csv`, 1326 Zeilen) - als
abschließenden Integrationstest der gesamten Arbeit.

**Ablauf:**
1. Duplikate erkennen und entfernen (Forschungsfrage 1)
2. Formatierungsfehler bei `price`, `saving`, den Datumsspalten beheben (Forschungsfrage 3)
3. Verbleibende echte Lücken bei `price` füllen (Forschungsfrage 2)
4. Verbleibende echte Lücken bei `parent_category` semantisch füllen (Forschungsfrage 4)
5. Vorher/Nachher-Datenqualität vergleichen und - soweit möglich - gegen
   `rfd_main_cleaned.csv` validieren

**Wichtiger methodischer Hinweis:** Alle Modelle werden hier, wo sinnvoll, auf der
**gesamten** verfügbaren Datenmenge (Benchmark train+val+test kombiniert bzw. alle
bekannten Werte des echten Rohdatensatzes) neu trainiert, um für die finale Anwendung
die maximale Datenmenge zu nutzen. Die in Kapitel 4.1-4.4 berichteten
Precision/Recall/F1/MAE/...-Werte basieren weiterhin ausschließlich auf den dortigen,
strikten Train/Test-Splits und werden durch dieses Notebook nicht verändert.


## 0. Methodenauswahl: Begründung anhand der Benchmark-Ergebnisse (Kapitel 4.1-4.4)

Für jeden Fehlertyp wird hier **eine konkrete** Methode aus dem jeweiligen
Dreiervergleich ausgewählt - nicht zwangsläufig die mit der höchsten Punktzahl,
sondern die unter Berücksichtigung von Qualität, Laufzeit und Kosten sinnvollste:

| Fehlerart | Gewählte Methode | Begründung |
|---|---|---|
| TF1 Duplikate | **Splink + regelbasierte Titel-Plausibilitätsprüfung** | Identische Qualität wie Claude (F1=0,986), aber ca. 10x schneller und ohne API-Kosten. Die zusätzliche Titelprüfung ist notwendig, siehe Abschnitt 3.3. |
| TF3 `price`-Format | **XGBoost + Parser** | Höchste Exact-Match-Rate aller drei Methoden (0,996). |
| TF3 `saving`-Format | **Regelbasiert (Regex + Parser)** | Höhere Exact-Match-Rate als der gelernte XGBoost-Klassifikator (0,980 vs. 0,946). |
| TF3 Datumsspalten | **Regelbasiert (Parser)** | Einzige sinnvolle Methode (deterministisches Rohformat, 1,000 Exact-Match). |
| TF2 `price`-Lücken | **Random-Forest-basierte Imputation nach dem MissForest-Prinzip** | Nahezu identische Güte wie Claude (RMSE-Differenz ≈ 2,3), aber kostenlos und ohne Laufzeitabhängigkeit von einer externen API. Keine vollständige iterative MissForest-Implementierung, siehe Abschnitt 5. |
| TF4 `parent_category`-Lücken | **Dictionary/Fuzzy-Mapping** | Klar beste Methode im Vergleich (Accuracy=0,895, Macro-F1=0,924, gegenüber 0,887/0,808 bei Claude und 0,710/0,448 bei Sentence Transformers), zugleich praktisch kostenlos. |

Der LLM-Ansatz wird für keinen der vier Fehlertypen als finale Methode gewählt: Er ist
in keinem einzigen Fall der klare Gewinner (bei TF1/TF3-Preis/TF2-Preis liegt er
höchstens gleichauf, bei TF2-Kategorie und TF4 unterliegt er den einfacheren
Verfahren), verursacht aber in jedem Fall die höchste Laufzeit und als einziger
Ansatz laufende API-Kosten. Dies ist selbst ein zentrales Ergebnis dieser Arbeit
(vgl. Kapitel 5).


## 1. Setup und Datenladen

In [1]:
import pandas as pd
import numpy as np
import re
import html
import json
import time
import math
import os
from collections import defaultdict
from difflib import SequenceMatcher
from urllib.parse import urlsplit, parse_qs, unquote

import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score

from splink import Linker, DuckDBAPI, SettingsCreator
import splink.comparison_library as cl
import splink.comparison_level_library as cll
from rapidfuzz import fuzz, process

SEED = 42
os.makedirs("results", exist_ok=True)
PIPELINE_T0 = time.time()

df_raw = pd.read_csv("data/rfd_main.csv").drop(columns=["Unnamed: 0"]).reset_index(drop=True)
df_raw["row_id"] = df_raw.index

df_clean = pd.read_csv("data/rfd_main_cleaned.csv").drop(columns=["Unnamed: 0"]).reset_index(drop=True)
df_clean["row_id"] = df_clean.index

row_mapping = pd.read_csv("benchmark/raw_clean_row_mapping.csv")

print(f"Rohdatensatz: {df_raw.shape[0]} Zeilen, {df_raw.shape[1] - 1} Spalten")
print(f"Referenzdatensatz (bereinigt): {df_clean.shape[0]} Zeilen")
print(f"Content-basierte Zuordnung Roh<->Referenz: {len(row_mapping)} Zeilen ({len(row_mapping) / len(df_raw):.1%})")


Rohdatensatz: 1326 Zeilen, 14 Spalten
Referenzdatensatz (bereinigt): 1325 Zeilen
Content-basierte Zuordnung Roh<->Referenz: 1293 Zeilen (97.5%)


## 2. Datenqualität VORHER

In [2]:
def quality_snapshot(df, cols, label):
    print(f"--- Datenqualität {label} ({len(df)} Zeilen) ---")
    for col in cols:
        n_missing = df[col].isna().sum()
        print(f"  fehlend {col:18s}: {n_missing:4d} ({n_missing / len(df):.1%})")

QUALITY_COLS_RAW = ["price", "saving", "parent_category", "source", "url"]
quality_snapshot(df_raw, QUALITY_COLS_RAW, "VORHER (Rohdatensatz)")


--- Datenqualität VORHER (Rohdatensatz) (1326 Zeilen) ---
  fehlend price             :  428 (32.3%)
  fehlend saving            :  802 (60.5%)
  fehlend parent_category   :  501 (37.8%)
  fehlend source            :  339 (25.6%)
  fehlend url               :  282 (21.3%)


## 3. Schritt 1 - Forschungsfrage 1: Duplikate erkennen und entfernen (Splink + Titel-Plausibilitätsprüfung)

Anders als beim ursprünglich in diesem Notebook verwendeten, hier aber verworfenen
XGBoost-Klassifikator (siehe Begründung in Abschnitt 0) benötigt Splink **keine**
manuell konstruierten Blocking-Regeln und **keine** gelabelten Trainingspaare: Die
m-/u-Wahrscheinlichkeiten werden per Expectation-Maximization direkt aus der
Datenstruktur der 1326 echten Zeilen geschätzt (identische Methodik wie in
`Forschungsfrage_1_Duplikate_Splink.ipynb`, dort auf dem 1476-zeiligen
Benchmark-Pool). Bei $\binom{1326}{2} \approx 878{.}000$ möglichen Paaren ist die
Blocking-Regel `"1=1"` (alle Paare) für DuckDB weiterhin unproblematisch.


In [3]:
pool = pd.read_csv("benchmark/tf1_duplicate_pool.csv")
real_pool = pool[~pool["is_synthetic"]].reset_index(drop=True)
assert set(real_pool["row_id"]) == set(df_raw["row_id"]), "Pool und Rohdatensatz sollten dieselben row_ids haben."
print(f"Reale Zeilen fuer Duplikaterkennung: {len(real_pool)}")


Reale Zeilen fuer Duplikaterkennung: 1326


### 3.1 Merkmalsvorbereitung (identisch zu `Forschungsfrage_1_Duplikate_Splink.ipynb`)

In [4]:
def normalize_url(u):
    if pd.isna(u):
        return None
    s = html.unescape(str(u).strip())
    if not s:
        return None
    parts = urlsplit(s)
    qs = parse_qs(parts.query)
    for key in ("location", "url", "u", "q", "dest"):
        if key in qs and qs[key]:
            inner = unquote(qs[key][0])
            if inner and inner != s:
                return normalize_url(inner)
    path = parts.path.rstrip("/")
    return f"{parts.netloc}{path}".lower()

def extract_price_num(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d+(?:\.\d+)?)", str(val))
    return float(m.group(1)) if m else np.nan

def parse_pool_date(s):
    if pd.isna(s):
        return pd.NaT
    s2 = re.sub(r"(\d+)(st|nd|rd|th)", r"\1", str(s))
    try:
        return pd.to_datetime(s2, format="%b %d, %Y %I:%M %p")
    except ValueError:
        return pd.to_datetime(s2, errors="coerce")

real_pool["url_norm"] = real_pool["url"].apply(normalize_url)
real_pool["title_lower"] = real_pool["title"].astype(str).str.lower()
real_pool["price_num"] = real_pool["price"].apply(extract_price_num)
real_pool["creation_date_iso"] = real_pool["creation_date"].apply(parse_pool_date).dt.strftime("%Y-%m-%d %H:%M:%S")


### 3.2 Splink-Modell trainieren und auf allen echten Kandidatenpaaren anwenden

Dieselben fünf Vergleichsfelder und Settings wie im Benchmark-Notebook, hier jedoch
neu auf den 1326 echten Zeilen trainiert (statt auf dem gemischten 1476-zeiligen
Benchmark-Pool) - für die finale Anwendung soll das Modell die tatsächliche
Struktur der echten Daten lernen.


In [5]:
settings = SettingsCreator(
    link_type="dedupe_only",
    unique_id_column_name="row_id",
    comparisons=[
        cl.ExactMatch("url_norm"),
        cl.JaroWinklerAtThresholds("title_lower", [0.9, 0.7]),
        cl.ExactMatch("author"),
        cl.CustomComparison(
            output_column_name="price_num",
            comparison_levels=[
                cll.NullLevel("price_num"),
                cll.ExactMatchLevel("price_num"),
                cll.AbsoluteDifferenceLevel("price_num", 0.01),
                cll.ElseLevel(),
            ],
        ),
        cl.AbsoluteTimeDifferenceAtThresholds(
            "creation_date_iso", input_is_string=True, metrics="hour", thresholds=[72],
            datetime_format="%Y-%m-%d %H:%M:%S",
        ),
    ],
    blocking_rules_to_generate_predictions=["1=1"],
    retain_intermediate_calculation_columns=True,
)

db_api = DuckDBAPI()
linker = Linker(real_pool, settings, db_api=db_api)

t0 = time.time()
linker.training.estimate_u_using_random_sampling(max_pairs=1e6, seed=SEED)
linker.training.estimate_parameters_using_expectation_maximisation("1=1")
splink_train_time = time.time() - t0
print(f"Splink-Trainingszeit (u-Schaetzung + EM) auf {len(real_pool)} echten Zeilen: {splink_train_time:.2f}s")

t1 = time.time()
predictions = linker.inference.predict()
df_pred = predictions.as_pandas_dataframe()
splink_predict_time = time.time() - t1
print(f"Vorhersagezeit ({len(df_pred)} Kandidatenpaare): {splink_predict_time:.2f}s")

SPLINK_THRESHOLD = 0.01  # auf dem val-Split in Forschungsfrage_1_Duplikate_Splink.ipynb kalibriert
raw_candidates = df_pred[df_pred["match_probability"] >= SPLINK_THRESHOLD].copy()
print(f"Rohkandidaten (match_probability >= {SPLINK_THRESHOLD}): {len(raw_candidates)} von {len(df_pred)} Paaren")
print(df_pred["match_probability"].describe())


You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.


----- Estimating u probabilities using random sampling -----



Estimated u probabilities using random sampling



Your model is not yet fully trained. Missing estimates for:
    - url_norm (no m values are trained).
    - title_lower (no m values are trained).
    - author (no m values are trained).
    - price_num (no m values are trained).
    - creation_date_iso (no m values are trained).



----- Starting EM training session -----



...

Blocking time: 0.00 seconds


Splink-Trainingszeit (u-Schaetzung + EM) auf 1326 echten Zeilen: 6.11s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Predict time: 2.14 seconds


Vorhersagezeit (878475 Kandidatenpaare): 7.53s
Rohkandidaten (match_probability >= 0.01): 449 von 878475 Paaren
count    8.784750e+05
mean     1.175581e-04
std      8.921567e-03
min      5.375293e-23
25%      1.254063e-06
50%      3.528521e-06
75%      6.909728e-06
max      1.000000e+00
Name: match_probability, dtype: float64


### 3.3 Wichtiger methodischer Befund: dasselbe Verteilungsverschiebungs-Problem wie beim (verworfenen) XGBoost-Ansatz

Eine Stichprobenprüfung der 449 Rohkandidaten bei Schwelle 0,01 zeigt dasselbe Muster,
das bereits beim ursprünglich in diesem Notebook verwendeten XGBoost-Klassifikator
beobachtet wurde: Ein erheblicher Teil der Paare mit hoher `match_probability` sind
**keine** echten Duplikate, sondern zufällig ähnliche, aber inhaltlich verschiedene
Angebote (z. B. wurden eine "Kobalt"-Gehrungssäge und ein "Hamilton Beach"-Mixer mit
`match_probability = 0,898` als Duplikat eingestuft, ebenso zwei verschiedene
Logitech-Mäuse, verschiedene SSD-Modelle und zwei unterschiedliche Videospiele bei
`match_probability = 0,999`). Da Splinks Vergleichsfelder u. a. auf `price_num` und
`creation_date_iso` beruhen, reicht ein zufällig übereinstimmender Preis und ein enges
Zeitfenster bereits aus, um eine hohe `match_probability` zu erzeugen - unabhängig
davon, ob die Titel tatsächlich dasselbe Produkt beschreiben. Dieser Befund bestätigt,
dass die in Abschnitt 3.2 des XGBoost-Versuchs beobachtete Schwierigkeit **kein
methodenspezifisches Artefakt** von XGBoost war, sondern ein grundsätzliches Problem
der Anwendung eines auf einem "leichten" Benchmark kalibrierten Duplikaterkennungs-
Modells auf echte, im Schnitt deutlich schwierigere Kandidatenpaare.

**Korrektur für die finale Anwendung:** Analog zum XGBoost-Versuch wird die
Splink-Wahrscheinlichkeit mit einer regelbasierten Titel-Plausibilitätsprüfung
kombiniert (identische Schwellenwerte wie dort, für Vergleichbarkeit):

```
Duplikat, wenn:  match_probability >= 0.01   UND   (title_seq_sim >= 0.90 ODER title_jaccard >= 0.80)
```

Dies reduziert die 449 Rohkandidaten auf 20 hochplausible Paare (siehe Sichtung
unten) - fast ausschließlich identische oder nahezu identische Titel (Reposts),
teils mit geringfügig abweichendem Preis. Dass auch ein aus den Daten selbst
gelerntes, unüberwachtes Modell wie Splink ohne eine zusätzliche inhaltliche
Prüfung nicht zuverlässig zwischen echten Reposts und zufällig ähnlichen,
verschiedenen Angeboten unterscheiden kann, ist selbst ein Ergebnis dieser Arbeit
(vgl. Kapitel 5.3).


In [6]:
def title_similarity(a, b):
    a, b = str(a).lower(), str(b).lower()
    return SequenceMatcher(None, a, b).ratio()

def token_jaccard(a, b):
    ta, tb = set(re.findall(r"[a-z0-9]+", str(a).lower())), set(re.findall(r"[a-z0-9]+", str(b).lower()))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)

pool_idx = real_pool.set_index("row_id")
raw_candidates["title_a"] = raw_candidates["row_id_l"].astype(int).map(pool_idx["title"])
raw_candidates["title_b"] = raw_candidates["row_id_r"].astype(int).map(pool_idx["title"])
raw_candidates["title_sim"] = [title_similarity(a, b) for a, b in zip(raw_candidates["title_a"], raw_candidates["title_b"])]
raw_candidates["title_jac"] = [token_jaccard(a, b) for a, b in zip(raw_candidates["title_a"], raw_candidates["title_b"])]

GATE_MIN_TITLE_SIM = 0.90
GATE_MIN_TITLE_JACCARD = 0.80
predicted_dup_pairs = raw_candidates[
    (raw_candidates["title_sim"] >= GATE_MIN_TITLE_SIM) | (raw_candidates["title_jac"] >= GATE_MIN_TITLE_JACCARD)
][["row_id_l", "row_id_r", "match_probability", "title_a", "title_b"]].rename(
    columns={"row_id_l": "row_id_a", "row_id_r": "row_id_b"})

print(f"Nach kombiniertem Kriterium als Duplikat eingestuft: {len(predicted_dup_pairs)} Paare (statt {len(raw_candidates)} bei alleiniger Splink-Schwelle).")
for _, r in predicted_dup_pairs.sort_values("match_probability").iterrows():
    print(f"  [{r['match_probability']:.4f}] '{r['title_a']}'  <->  '{r['title_b']}'")


Nach kombiniertem Kriterium als Duplikat eingestuft: 20 Paare (statt 449 bei alleiniger Splink-Schwelle).
  [0.0198] 'Tribit XFree Tune Bluetooth Headphones Over-Ear - $45'  <->  'Tribit XFree Tune Bluetooth Headphones Over-Ear $40'
  [0.0718] 'One Step Hand Sanitizer, Fragrance-Free, 473mL, $6.99'  <->  'One Step Hand Sanitizer, Fragrance-Free, 1L, $9.98'
  [0.0961] 'Ooma Telo Home phone service device $99.99'  <->  'Ooma Telo Home phone service device $99.99'
  [0.9407] '(YMMV) Phillips LED Recessed Retrofit Trim 5-6" - $4'  <->  'Philips LED recessed retrofit trim 5-6" - $4'
  [1.0000] 'Napoleon BBQ - Extreme Saving Events - Up to $150 Mail Rebate - Effective May 15 - July 4'  <->  'Napoleon BBQ - Extreme Saving Events - Up to $150 Mail Rebate - Effective May 15 - July 4'
  [1.0000] 'Rakuten "Beat the Heat" Event: 10% CASH BACK!'  <->  'Rakuten "Beat the Heat" Event: 10% CASH BACK!'
  [1.0000] 'Book Outlet - Many Low Prices on Books (Free Shipping Over $45 & 16% Off)'  <->  'Book Ou

### 3.4 Duplikat-Cluster bilden und Repräsentanten auswählen

Union-Find über die als Duplikat eingestuften Paare; aus jedem Cluster wird die
**vollständigste** Zeile (wenigste fehlende Werte in `price`/`saving`/
`parent_category`/`source`/`url`) als Repräsentant behalten, bei Gleichstand die
mit der kleinsten `row_id`.


In [7]:
parent = {rid: rid for rid in real_pool["row_id"]}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[rx] = ry

for _, r in predicted_dup_pairs.iterrows():
    union(int(r["row_id_a"]), int(r["row_id_b"]))

clusters = defaultdict(list)
for rid in real_pool["row_id"]:
    clusters[find(rid)].append(rid)
dup_clusters = {root: sorted(members) for root, members in clusters.items() if len(members) > 1}

def completeness_score(rid):
    row = df_raw.loc[rid]
    return sum(pd.notna(row[c]) for c in ["price", "saving", "parent_category", "source", "url"])

removed_row_ids, cluster_report = [], []
for root, members in dup_clusters.items():
    best = max(members, key=lambda r: (completeness_score(r), -r))
    removed = [m for m in members if m != best]
    removed_row_ids.extend(removed)
    cluster_report.append({"cluster_id": root, "representative_row_id": best,
                            "removed_row_ids": removed, "cluster_size": len(members)})

print(f"Duplikat-Cluster gefunden: {len(dup_clusters)}")
print(f"Zu entfernende Zeilen: {len(removed_row_ids)} von {len(real_pool)} ({len(removed_row_ids) / len(real_pool):.1%})")

pd.DataFrame(cluster_report).to_csv("results/final_tf1_duplicate_clusters.csv", index=False)
removed_row_ids_set = set(removed_row_ids)
tf1_wall_time = splink_train_time + splink_predict_time


Duplikat-Cluster gefunden: 20
Zu entfernende Zeilen: 20 von 1326 (1.5%)


## 4. Schritt 2 - Forschungsfrage 3: Formatierungsfehler beheben

`price` wird weiterhin über den gelernten XGBoost-Formatklassifikator + Parser
normalisiert (beste Methode für `price`, vgl. Abschnitt 0). `saving` wird dagegen
**regelbasiert** (Regex-Kaskade, keine gelernte Formatklassifikation) behandelt, da
dies im Vergleich in Forschungsfrage 3 die höhere Exact-Match-Rate erzielte. Die
Datumsspalten werden - wie durchgängig in dieser Arbeit - ausschließlich über den
deterministischen Parser normalisiert.


In [8]:
def string_features(s):
    s = str(s)
    return {
        "length": len(s), "has_dollar": int("$" in s), "has_percent": int("%" in s),
        "has_alpha": int(bool(re.search(r"[A-Za-z]", s))), "has_slash": int("/" in s),
        "has_dash": int("-" in s), "n_digit_groups": len(re.findall(r"\d+", s)),
        "starts_with_digit": int(s[:1].isdigit()), "has_off_word": int("off" in s.lower()),
    }

FMT_FEAT_COLS = ["length", "has_dollar", "has_percent", "has_alpha", "has_slash",
                 "has_dash", "n_digit_groups", "starts_with_digit", "has_off_word"]

def extract_first_number(s):
    m = re.search(r"(\d+(?:\.\d+)?)", str(s))
    return float(m.group(1)) if m else np.nan

def parse_price_by_class(raw_value, format_class):
    s = str(raw_value)
    num = extract_first_number(s)
    if pd.isna(num):
        return np.nan
    if format_class == "prozent":
        return np.nan
    if format_class == "preisspanne":
        nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", s)]
        return float(np.mean(nums)) if nums else np.nan
    if format_class == "wortwert":
        return 0.0 if "free" in s.lower() else np.nan
    return num

def parse_raw_date(s):
    if pd.isna(s):
        return pd.NaT
    s2 = re.sub(r"(\d+)(st|nd|rd|th)", r"\1", str(s))
    for fmt in ("%b %d, %Y %I:%M %p", "%b %d, %Y", "%B %d, %Y %I:%M %p", "%B %d, %Y"):
        try:
            return pd.to_datetime(s2, format=fmt)
        except ValueError:
            continue
    return pd.NaT

def train_full_format_classifier(bench_path):
    bench = pd.read_csv(bench_path)  # train+val+test zusammen: maximale Datenmenge fuer die finale Anwendung
    X = pd.DataFrame([string_features(v) for v in bench["raw_value"]])[FMT_FEAT_COLS]
    le = LabelEncoder()
    y = le.fit_transform(bench["format_class"])
    clf = xgb.XGBClassifier(n_estimators=150, max_depth=4, random_state=SEED, eval_metric="mlogloss")
    clf.fit(X, y)
    return clf, le

def classify_and_parse(raw_series, clf, le, parser_fn):
    out = pd.Series(np.nan, index=raw_series.index, dtype="object")
    known = raw_series.notna()
    if known.sum() == 0:
        return out
    X = pd.DataFrame([string_features(v) for v in raw_series[known]])[FMT_FEAT_COLS]
    classes = le.inverse_transform(clf.predict(X))
    out.loc[known] = [parser_fn(v, c) for v, c in zip(raw_series[known], classes)]
    return pd.to_numeric(out, errors="coerce")

def parse_saving_ratio_regelbasiert(raw_value, cleaned_price):
    """Regelbasiert (Regex), identisch zu Forschungsfrage_3_Formatierung_Regelbasiert.ipynb -
    verwendet den bereits per XGBoost+Parser bereinigten eigenen price_clean-Wert als Basis
    fuer die Ratio-Formel."""
    s = str(raw_value)
    if "%" in s:
        m = re.search(r"(\d+(?:\.\d+)?)\s*%", s)
        return float(m.group(1)) / 100.0 if m else np.nan
    amount = extract_first_number(s)
    if pd.isna(amount) or pd.isna(cleaned_price):
        return np.nan
    denom = cleaned_price + amount
    return amount / denom if denom > 0 else np.nan

t0 = time.time()
clf_price, le_price = train_full_format_classifier("benchmark/tf3_format_price.csv")

df_final = df_raw.copy()
df_final["price_clean"] = classify_and_parse(df_raw["price"], clf_price, le_price, parse_price_by_class)

df_final["saving_clean"] = [
    parse_saving_ratio_regelbasiert(raw, own_price)
    for raw, own_price in zip(df_raw["saving"], df_final["price_clean"])
]

df_final["creation_date_clean"] = df_raw["creation_date"].apply(parse_raw_date)
df_final["expiry_clean"] = df_raw["expiry"].apply(parse_raw_date)
df_final["last_reply_clean"] = df_raw["last_reply"].apply(parse_raw_date)
tf3_time = time.time() - t0

print(f"Formatierung angewendet in {tf3_time:.2f}s.")
print(f"  price_clean  (XGBoost+Parser) : {df_final['price_clean'].notna().sum()}/{df_raw['price'].notna().sum()} vorhandene Rohwerte erfolgreich geparst")
print(f"  saving_clean (Regelbasiert)   : {df_final['saving_clean'].notna().sum()}/{df_raw['saving'].notna().sum()} vorhandene Rohwerte erfolgreich geparst")
print(f"  Datumsspalten (Regelbasiert)  : {df_final['creation_date_clean'].notna().sum()}/{len(df_raw)} creation_date, "
      f"{df_final['expiry_clean'].notna().sum()}/{df_raw['expiry'].notna().sum()} expiry, "
      f"{df_final['last_reply_clean'].notna().sum()}/{len(df_raw)} last_reply")


Formatierung angewendet in 0.36s.
  price_clean  (XGBoost+Parser) : 874/898 vorhandene Rohwerte erfolgreich geparst
  saving_clean (Regelbasiert)   : 497/524 vorhandene Rohwerte erfolgreich geparst
  Datumsspalten (Regelbasiert)  : 1326/1326 creation_date, 370/370 expiry, 1326/1326 last_reply


## 5. Schritt 3 - Forschungsfrage 2: Verbleibende echte Lücken bei `price` füllen (Random-Forest-basierte Imputation nach dem MissForest-Prinzip)

Nach Schritt 2 ist `price_clean` für alle Zeilen mit vorhandenem Rohwert gefüllt. Für
die Zeilen, in denen `price` im Rohdatensatz **echt fehlt**, wird derselbe
`RandomForestRegressor`-Fit wie in `Forschungsfrage_2_FehlendeWerte_MissForest.ipynb`
angewendet - identisches Feature-Set (`parent_category`, `thread_category`, `source`,
`views`, `votes`, `replies`), hier trainiert auf allen Zeilen des echten Rohdatensatzes
mit bekanntem Preis (nicht nur der Benchmark-Stichprobe). Wie dort erläutert, handelt
es sich um eine Random-Forest-basierte Imputation nach dem MissForest-**Prinzip**,
nicht um dessen vollständige iterative Implementierung; fehlende Werte in den
kategorialen Hilfsmerkmalen (u. a. `parent_category`, das an dieser Stelle der
Pipeline selbst noch echte Lücken enthält) werden vorab per Platzhalterkategorie
behandelt.

Für **201 dieser Zeilen** existiert über die Content-Zuordnung zu `rfd_main_cleaned.csv`
ein echter Referenzwert - das erlaubt eine **echte**, nicht-synthetische Validierung
dieses Schritts.


In [9]:
missing_price_mask = df_final["price_clean"].isna()
print(f"Echte Preis-Luecken im Rohdatensatz: {missing_price_mask.sum()}")

cat_cols_p = ["parent_category", "thread_category", "source"]
num_cols_p = ["views", "votes", "replies"]

ctx = df_final[cat_cols_p].fillna("__missing__")
encoder_p = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
known_price_mask = ~missing_price_mask
encoder_p.fit(ctx.loc[known_price_mask])

def build_X_price(mask):
    return np.hstack([df_final.loc[mask, num_cols_p].values, encoder_p.transform(ctx.loc[mask])])

t0 = time.time()
rf_price_final = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)
rf_price_final.fit(build_X_price(known_price_mask), df_final.loc[known_price_mask, "price_clean"])
tf2_price_time = time.time() - t0

imputed_price = rf_price_final.predict(build_X_price(missing_price_mask))
df_final.loc[missing_price_mask, "price_clean"] = imputed_price
print(f"{missing_price_mask.sum()} Preise per Random-Forest-Imputation nach dem MissForest-Prinzip imputiert (Trainingszeit {tf2_price_time:.2f}s).")


Echte Preis-Luecken im Rohdatensatz: 452


452 Preise per Random-Forest-Imputation nach dem MissForest-Prinzip imputiert (Trainingszeit 1.26s).


### 5.1 Echte Validierung der Preis-Imputation gegen die 201 Zeilen mit Referenzwert

In [10]:
r = df_raw[["row_id", "price"]].rename(columns={"price": "price_raw"})
c = df_clean[["row_id", "price"]].rename(columns={"price": "price_clean_reference"})
val = row_mapping.merge(r, left_on="row_id_raw", right_on="row_id").merge(c, left_on="row_id_clean", right_on="row_id")
val = val[val["price_raw"].isna() & val["price_clean_reference"].notna()]
val = val.merge(df_final[["row_id", "price_clean"]], left_on="row_id_raw", right_on="row_id", suffixes=("", "_pipeline"))

if len(val) > 0:
    mae_real = mean_absolute_error(val["price_clean_reference"], val["price_clean"])
    rmse_real = np.sqrt(mean_squared_error(val["price_clean_reference"], val["price_clean"]))
    print(f"Echte Preis-Imputations-Validierung (n={len(val)}): MAE={mae_real:.2f}  RMSE={rmse_real:.2f}")
    print("(Vergleichswert Forschungsfrage 2, synthetische Maskierung: siehe results/tf2_missforest_metrics.json)")
else:
    mae_real = rmse_real = None
    print("Keine Zeilen mit echtem Referenzwert gefunden.")


Echte Preis-Imputations-Validierung (n=201): MAE=159.09  RMSE=290.97
(Vergleichswert Forschungsfrage 2, synthetische Maskierung: siehe results/tf2_missforest_metrics.json)


## 6. Schritt 4 - Forschungsfrage 4: Verbleibende echte Lücken bei `parent_category` füllen (Dictionary/Fuzzy)

Für die reale Lücke (Zeilen ohne `parent_category`) wird das **Dictionary/Fuzzy-Mapping**
aus Forschungsfrage 4 verwendet - die dort empirisch beste Methode (vgl. Abschnitt 0).
Die Mehrheitsvotum-Tabelle (`thread_category` -> häufigste zugehörige
`parent_category`) wird hier direkt aus den 825 Zeilen des echten Rohdatensatzes mit
bekannter `parent_category` aufgebaut - nicht aus der separaten Benchmark-Stichprobe -,
da diese echten Paare die maximale und realistischste verfügbare Datenmenge für die
finale Anwendung darstellen. Für unbekannte `thread_category`-Werte greift RapidFuzz
(Schwelle 80) mit demselben Sonderfall- und Rückfall-Verhalten wie im
Benchmark-Notebook; fehlt `thread_category` selbst, wird direkt auf die insgesamt
häufigste Kategorie zurückgefallen - ein separates Fallback-Modell (wie zuvor mit
TF-IDF+RF) ist damit nicht mehr nötig.

**Wichtige Einschränkung:** Der Referenzdatensatz `rfd_main_cleaned.csv` füllt diese
Lücke ebenfalls nicht - für diesen Schritt existiert also keine Ground Truth, die
eine quantitative Validierung erlauben würde. Die Bewertung bleibt auf die in
Forschungsfrage 4 berichtete Benchmark-Genauigkeit beschränkt; hier wird nur die
reale Anwendung demonstriert.


In [11]:
missing_cat_mask = df_final["parent_category"].isna()
print(f"Echte parent_category-Luecken im Rohdatensatz: {missing_cat_mask.sum()}")

known_cat_df = df_final[df_final["parent_category"].notna()]
VALID_CATEGORIES_FINAL = sorted(known_cat_df["parent_category"].unique().tolist())
majority_map_final = known_cat_df.groupby("thread_category")["parent_category"].agg(lambda s: s.value_counts().idxmax())
global_majority_final = known_cat_df["parent_category"].value_counts().idxmax()
known_thread_categories_final = list(majority_map_final.index)
FUZZY_THRESHOLD = 80

print(f"Bekannte thread_category-Werte (825 reale Zeilen mit bekannter parent_category): {len(majority_map_final)}")
print(f"Globale Rueckfall-Kategorie: {global_majority_final}")

def map_thread_to_parent_final(thread_category):
    if pd.isna(thread_category):
        return global_majority_final, "kein_wert"
    if thread_category in majority_map_final.index:
        return majority_map_final[thread_category], "exakt"
    if thread_category in VALID_CATEGORIES_FINAL:
        return thread_category, "sonderfall_bereits_parent_category"
    match = process.extractOne(thread_category, known_thread_categories_final, scorer=fuzz.token_sort_ratio)
    if match is not None and match[1] >= FUZZY_THRESHOLD:
        matched = match[0]
        return majority_map_final[matched], f"fuzzy(score={match[1]:.0f} -> '{matched}')"
    return global_majority_final, "globaler_rueckfall"

t0 = time.time()
mapped_final = df_final.loc[missing_cat_mask, "thread_category"].apply(map_thread_to_parent_final)
df_final.loc[missing_cat_mask, "parent_category_clean"] = mapped_final.apply(lambda x: x[0])
df_final.loc[missing_cat_mask, "zuordnungsart"] = mapped_final.apply(lambda x: x[1])
df_final.loc[~missing_cat_mask, "parent_category_clean"] = df_final.loc[~missing_cat_mask, "parent_category"]
tf4_time = time.time() - t0

print(f"{missing_cat_mask.sum()} parent_category-Werte per Dictionary/Fuzzy-Mapping gefuellt ({tf4_time*1000:.2f}ms).")
print(df_final.loc[missing_cat_mask, "zuordnungsart"].value_counts())


Echte parent_category-Luecken im Rohdatensatz: 501
Bekannte thread_category-Werte (825 reale Zeilen mit bekannter parent_category): 42
Globale Rueckfall-Kategorie: Computers & Electronics
501 parent_category-Werte per Dictionary/Fuzzy-Mapping gefuellt (4.61ms).
zuordnungsart
sonderfall_bereits_parent_category    423
globaler_rueckfall                     77
kein_wert                               1
Name: count, dtype: int64


## 7. Zusammenführung: finaler bereinigter Datensatz

In [12]:
df_output = df_final.drop(columns=["row_id"]).copy()
df_output.insert(0, "row_id", df_final["row_id"])
df_output["war_duplikat_entfernt"] = df_output["row_id"].isin(removed_row_ids_set)

df_final_clean = df_output[~df_output["row_id"].isin(removed_row_ids_set)].reset_index(drop=True)

OUTPUT_COLS = ["row_id", "author", "creation_date_clean", "expiry_clean", "last_reply_clean",
               "parent_category_clean", "price_clean", "replies", "saving_clean", "source",
               "thread_category", "title", "url", "views", "votes"]
df_final_clean[OUTPUT_COLS].to_csv("results/rfd_main_final_cleaned_by_pipeline.csv", index=False)
df_output.to_csv("results/rfd_main_final_with_duplicate_flags.csv", index=False)

print(f"Finaler bereinigter Datensatz: {len(df_final_clean)} Zeilen "
      f"({len(removed_row_ids_set)} als Duplikat entfernt von urspruenglich {len(df_raw)}).")


Finaler bereinigter Datensatz: 1306 Zeilen (20 als Duplikat entfernt von urspruenglich 1326).


## 8. Datenqualität NACHHER und Vergleich mit dem Referenzdatensatz

In [13]:
QUALITY_COLS_CLEAN = ["price_clean", "saving_clean", "parent_category_clean", "source", "url"]
quality_snapshot(df_final_clean, QUALITY_COLS_CLEAN, "NACHHER (bereinigt durch diese Pipeline)")
print()
print("Hinweis: 'fehlend saving' bleibt bewusst hoch - eine fehlende Ersparnisangabe bedeutet in der Regel,")
print("dass fuer den betreffenden Deal schlicht kein Rabatt ausgewiesen wurde (kein Bereinigungsfehler),")
print("konsistent mit der Konvention im Referenzdatensatz rfd_main_cleaned.csv.")


--- Datenqualität NACHHER (bereinigt durch diese Pipeline) (1306 Zeilen) ---
  fehlend price_clean       :    0 (0.0%)
  fehlend saving_clean      :  816 (62.5%)
  fehlend parent_category_clean:    0 (0.0%)
  fehlend source            :  331 (25.3%)
  fehlend url               :  279 (21.4%)

Hinweis: 'fehlend saving' bleibt bewusst hoch - eine fehlende Ersparnisangabe bedeutet in der Regel,
dass fuer den betreffenden Deal schlicht kein Rabatt ausgewiesen wurde (kein Bereinigungsfehler),
konsistent mit der Konvention im Referenzdatensatz rfd_main_cleaned.csv.


### 8.1 Übereinstimmung mit `rfd_main_cleaned.csv` (soweit dort Referenzwerte vorhanden sind)

Für alle Zeilen, die per Content-Join einer Zeile in `rfd_main_cleaned.csv` zugeordnet
werden können, wird hier verglichen, wie nah der Output dieser vollständigen Pipeline
am professionell bereinigten Referenzdatensatz liegt - der abschließende
End-to-End-Test der gesamten Arbeit.


In [14]:
r2 = df_raw[["row_id", "price", "saving"]].rename(columns={"price": "price_raw", "saving": "saving_raw"})
c2 = df_clean[["row_id", "price", "saving", "creation_date"]].rename(
    columns={"price": "price_ref", "saving": "saving_ref", "creation_date": "creation_date_ref"})
cmp = row_mapping.merge(r2, left_on="row_id_raw", right_on="row_id").merge(c2, left_on="row_id_clean", right_on="row_id")
cmp = cmp.merge(df_final[["row_id", "price_clean", "saving_clean", "creation_date_clean"]],
                left_on="row_id_raw", right_on="row_id", suffixes=("", "_pipeline"))
cmp = cmp[~cmp["row_id_raw"].isin(removed_row_ids_set)]  # entfernte Duplikate aus dem Vergleich ausschliessen

price_cmp = cmp.dropna(subset=["price_ref", "price_clean"])
price_mae_full = mean_absolute_error(price_cmp["price_ref"], price_cmp["price_clean"])
print(f"price:  MAE={price_mae_full:.3f}  (n={len(price_cmp)} vergleichbare Zeilen, "
      f"davon {val.shape[0] if len(val) else 0} echte Luecken-Imputationen)")

saving_cmp = cmp.dropna(subset=["saving_ref", "saving_clean"])
saving_mae_full = (saving_cmp["saving_ref"] - saving_cmp["saving_clean"]).abs().mean()
print(f"saving: MAE={saving_mae_full:.4f}  (n={len(saving_cmp)} vergleichbare Zeilen)")

cmp["creation_date_ref_parsed"] = pd.to_datetime(cmp["creation_date_ref"], errors="coerce")
date_cmp = cmp.dropna(subset=["creation_date_ref_parsed", "creation_date_clean"])
date_match_full = (date_cmp["creation_date_clean"].dt.floor("min") == date_cmp["creation_date_ref_parsed"].dt.floor("min")).mean()
print(f"creation_date: Exact-Match-Rate={date_match_full:.3f}  (n={len(date_cmp)} vergleichbare Zeilen)")


price:  MAE=30.537  (n=1052 vergleichbare Zeilen, davon 201 echte Luecken-Imputationen)
saving: MAE=0.0000  (n=483 vergleichbare Zeilen)
creation_date: Exact-Match-Rate=1.000  (n=1289 vergleichbare Zeilen)


## 9. Zusammenfassung und Laufzeit-Log

In [15]:
pipeline_wall_time = time.time() - PIPELINE_T0

FINAL_METHODS = {
    "tf1_duplikate": "Splink + regelbasierte Titel-Plausibilitaetspruefung",
    "tf3_price": "XGBoost_Parser",
    "tf3_saving": "Regelbasiert",
    "tf3_datum": "Regelbasiert",
    "tf2_fehlende_werte_price": "RandomForest_nach_MissForest_Prinzip",
    "tf4_semantik": "Dictionary_Fuzzy",
}

summary = {
    "n_input_rows": int(len(df_raw)),
    "n_duplicates_removed": int(len(removed_row_ids_set)),
    "n_output_rows": int(len(df_final_clean)),
    "tf1_splink_train_time_sec": splink_train_time,
    "tf1_splink_predict_time_sec": splink_predict_time,
    "tf1_splink_threshold": SPLINK_THRESHOLD,
    "tf2_price_real_gap_mae": float(mae_real) if mae_real is not None else None,
    "tf2_price_real_gap_rmse": float(rmse_real) if rmse_real is not None else None,
    "price_mae_vs_reference_full": float(price_mae_full),
    "price_n_compared_full": int(len(price_cmp)),
    "saving_mae_vs_reference_full": float(saving_mae_full),
    "date_exact_match_vs_reference_full": float(date_match_full),
    "n_parent_category_filled": int(missing_cat_mask.sum()),
    "pipeline_wall_time_sec": pipeline_wall_time,
    "method_config": FINAL_METHODS,
}
with open("results/finale_bereinigung_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

log_row = pd.DataFrame([{
    "experiment": "Finale_Anwendung_Gesamtpipeline", "method": "Bester_Methodenmix", "n_items": len(df_raw),
    "wall_time_sec": pipeline_wall_time, "input_tokens": 0, "output_tokens": 0,
    "estimated_cost_usd": 0.0, "model_name": "splink+xgboost_parser+regelbasiert+randomforest_missforest_prinzip+dictionary_fuzzy",
}])
log_path = "results/laufzeit_kosten_log.csv"
if os.path.exists(log_path):
    _old_log = pd.read_csv(log_path)
    _new_keys = set(zip(log_row["experiment"], log_row["method"]))
    _old_log = _old_log[~_old_log.apply(lambda r: (r["experiment"], r["method"]) in _new_keys, axis=1)]
    # alte Zeile aus der vorherigen (inkonsistenten) Pipeline-Version ebenfalls entfernen
    _old_log = _old_log[~((_old_log["experiment"] == "Finale_Anwendung_Gesamtpipeline") & (_old_log["method"] == "Klassisch_Gesamt"))]
    _combined_log = pd.concat([_old_log, log_row], ignore_index=True)
else:
    _combined_log = log_row
_combined_log.to_csv(log_path, index=False)

print(json.dumps(summary, indent=2, default=str))
print()
print("Gespeichert: results/rfd_main_final_cleaned_by_pipeline.csv, "
      "results/rfd_main_final_with_duplicate_flags.csv, results/finale_bereinigung_summary.json")


{
  "n_input_rows": 1326,
  "n_duplicates_removed": 20,
  "n_output_rows": 1306,
  "tf1_splink_train_time_sec": 6.114808797836304,
  "tf1_splink_predict_time_sec": 7.5337138175964355,
  "tf1_splink_threshold": 0.01,
  "tf2_price_real_gap_mae": 159.09024991708122,
  "tf2_price_real_gap_rmse": 290.9684387215566,
  "price_mae_vs_reference_full": 30.536519518377688,
  "price_n_compared_full": 1052,
  "saving_mae_vs_reference_full": 4.903563255966743e-05,
  "date_exact_match_vs_reference_full": 1.0,
  "n_parent_category_filled": 501,
  "pipeline_wall_time_sec": 15.940563917160034,
  "method_config": {
    "tf1_duplikate": "Splink + regelbasierte Titel-Plausibilitaetspruefung",
    "tf3_price": "XGBoost_Parser",
    "tf3_saving": "Regelbasiert",
    "tf3_datum": "Regelbasiert",
    "tf2_fehlende_werte_price": "RandomForest_nach_MissForest_Prinzip",
    "tf4_semantik": "Dictionary_Fuzzy"
  }
}

Gespeichert: results/rfd_main_final_cleaned_by_pipeline.csv, results/rfd_main_final_with_duplicate_